# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import numpy as np
import pandas as pd
from itertools import product
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import yaml
import duckdb

import sys
sys.path.append(str(Path(globals()['_dh'][0]).resolve().parent.parent))

from generator.paths import project_path, pipeline_path, input_path, data_path, output_path
from generator.library.utilities import sort_params, clear_api, ktok
from generator.library.randomizer import randomize_demand
from generator.library.growth import generate_growth_time_series
from generator.library.loaders import base_loader_csv
from generator.transformers.dimension import segment_constant_factor_split, segment_add_total, segment_append_constant, segment_append_percentage, segment_append_reminder, segment_remove
from generator.library.db import read_sample, count_rows
import generator.library.scenario_constraints


In [2]:
# 2. Load configuration (new version, come back to this later)

with open(project_path / 'config.yaml', "r") as f:
    config = yaml.safe_load(f)

with open(pipeline_path / 'county-prototype.yaml', "r") as f:
    pipeline = yaml.safe_load(f)

## Set flag to clear the api folder
clear_api_flag = True

In [3]:
# 3. Calculate the scenarios (OLD)
'''
## Extract and load the constraint functions
constraint_names = [ scenario['name'] for scenario in config['scenarios'] if scenario['type'] == 'constraint' and scenario['disable'] == False ]
constraint_functions = [ getattr(library.scenario_constraints, name) for name in constraint_names ]

# Extract names and values from non-constraint parameters
parameter_objs = [obj for obj in config['scenarios'] if obj['type'] != 'constraint' and obj['disable'] == False]
names = [obj['name'] for obj in parameter_objs]
values = [[item['value'] for item in obj['items']] for obj in parameter_objs]

# Build default scenario
default_scenario = {
    obj["name"]: obj["default"]
    for obj in parameter_objs
    if "default" in obj
}

# Validate default scenario
if not all(fn(default_scenario) for fn in constraint_functions):
    raise ValueError(f"Default scenario violates constraints: {default_scenario}")

# Generate all scenarios
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Filter using constraints and mark the default
scenarios = []
for s in all_scenarios:
    if all(fn(s) for fn in constraint_functions):
        s_out = dict(s)
        if s == default_scenario:
            s_out["default"] = True
        scenarios.append(s_out)
'''

'\n## Extract and load the constraint functions\nconstraint_names = [ scenario[\'name\'] for scenario in config[\'scenarios\'] if scenario[\'type\'] == \'constraint\' and scenario[\'disable\'] == False ]\nconstraint_functions = [ getattr(library.scenario_constraints, name) for name in constraint_names ]\n\n# Extract names and values from non-constraint parameters\nparameter_objs = [obj for obj in config[\'scenarios\'] if obj[\'type\'] != \'constraint\' and obj[\'disable\'] == False]\nnames = [obj[\'name\'] for obj in parameter_objs]\nvalues = [[item[\'value\'] for item in obj[\'items\']] for obj in parameter_objs]\n\n# Build default scenario\ndefault_scenario = {\n    obj["name"]: obj["default"]\n    for obj in parameter_objs\n    if "default" in obj\n}\n\n# Validate default scenario\nif not all(fn(default_scenario) for fn in constraint_functions):\n    raise ValueError(f"Default scenario violates constraints: {default_scenario}")\n\n# Generate all scenarios\nall_scenarios = [dict(zip(

In [4]:
# 3. Calculate the scenarios

# Extract names and values
names = [obj['name'] for obj in config['scenario']['scenarios']]
values = [[item['value'] for item in obj['items']] for obj in config['scenario']['scenarios']]

# Build default scenario
default_scenario = {
    obj["name"]: obj["default"]
    for obj in config['scenario']['scenarios']
    if "default" in obj
}

# Generate all scenarios
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Filter using constraints and mark the default
scenarios = []
for s in all_scenarios:
    s_out = dict(s)
    if s == default_scenario:
        s_out["default"] = True
    scenarios.append(s_out)

In [5]:
pipeline['base']

{'file': 'base-demand/base-load-curve,aggregation=mean,base-year=2025,geography=SE,normalized=False,resolution=1h.csv',
 'index': 'timestamp',
 'segments': []}

In [6]:
# 3. Load the base demand

base_loader_csv(pipeline['base'], input_path / pipeline['base']['file'], output_path / pipeline['database']['file'], pipeline['database']['defaultTable'])

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core',
 'added_rows': 8760,
 'added_columns': ['timestamp', 'value']}

In [7]:
read_sample(output_path / pipeline['database']['file'], pipeline['database']['defaultTable'], 10)

,timestamp,value
0,2024-01-01 00:00:00,13609.395626
1,2024-01-01 01:00:00,13581.126613
2,2024-01-01 02:00:00,13513.944426
3,2024-01-01 03:00:00,13443.488412
4,2024-01-01 04:00:00,13530.047135
5,2024-01-01 05:00:00,13730.022163
6,2024-01-01 06:00:00,14212.371650
7,2024-01-01 07:00:00,14604.052946
8,2024-01-01 08:00:00,14797.331828
9,2024-01-01 09:00:00,14963.197503


In [8]:
# 4 Execute transformations - Sort the transformations array

transformers = pipeline['transformers']
transformers.sort(key=lambda x: x["order"])

In [9]:
# 4.1. Execute transformations - dimensionalize over geographies

transformer = transformers[0]
inputs = transformer['inputs']
geographies = pd.read_csv(input_path / inputs[0]['path'], dtype={"geography": str, "factor": float})

segment_constant_factor_split(output_path / pipeline['database']['file'], pipeline['database']['defaultTable'], f"{pipeline['database']['defaultTable']}_geographies", geographies)


{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_geographies',
 'rows': 175200,
 'columns': ['geography']}

In [10]:
# 4.2 Execute transformations - dimensionalize over segments

transformer = transformers[1]
inputs = transformer['inputs']

In [11]:
segment_add_total(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_geographies", f"{pipeline['database']['defaultTable']}_segments", 'sector')

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_segments',
 'rows': 0,
 'columns': ['sector']}

In [12]:
read_sample(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", 10)

,timestamp,geography,value,sector
0,2024-01-01 00:00:00,01,1623.222880,total
1,2024-01-01 01:00:00,01,1619.851172,total
2,2024-01-01 02:00:00,01,1611.838203,total
3,2024-01-01 03:00:00,01,1603.434758,total
4,2024-01-01 04:00:00,01,1613.758809,total
5,2024-01-01 05:00:00,01,1637.610275,total
6,2024-01-01 06:00:00,01,1695.141171,total
7,2024-01-01 07:00:00,01,1741.857870,total
8,2024-01-01 08:00:00,01,1764.910672,total
9,2024-01-01 09:00:00,01,1784.693840,total


In [13]:
segment_append_constant(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", f"{pipeline['database']['defaultTable']}_segments", 'sector', 'industry', 100)

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_segments',
 'rows': 183960,
 'columns': []}

In [14]:
read_sample(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", where="sector='industry'")

,timestamp,geography,sector,value
0,2024-01-01 00:00:00,01,industry,100.0
1,2024-01-01 01:00:00,01,industry,100.0
2,2024-01-01 02:00:00,01,industry,100.0
3,2024-01-01 03:00:00,01,industry,100.0
4,2024-01-01 04:00:00,01,industry,100.0
...,...,...,...,...
183955,2024-12-31 19:00:00,25,industry,100.0
183956,2024-12-31 20:00:00,25,industry,100.0
183957,2024-12-31 21:00:00,25,industry,100.0
183958,2024-12-31 22:00:00,25,industry,100.0


In [15]:
segment_append_percentage(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", f"{pipeline['database']['defaultTable']}_segments", 'sector', 'total', 'transport', 0.15)

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_segments',
 'rows': 183960,
 'columns': []}

In [16]:
read_sample(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", where="sector='transport'")

,timestamp,geography,sector,value
0,2024-01-01 00:00:00,01,transport,243.483432
1,2024-01-01 01:00:00,01,transport,242.977676
2,2024-01-01 02:00:00,01,transport,241.775730
3,2024-01-01 03:00:00,01,transport,240.515214
4,2024-01-01 04:00:00,01,transport,242.063821
...,...,...,...,...
183955,2024-12-31 19:00:00,25,transport,196.176126
183956,2024-12-31 20:00:00,25,transport,191.323100
183957,2024-12-31 21:00:00,25,transport,188.869610
183958,2024-12-31 22:00:00,25,transport,186.025903


In [17]:
segment_append_reminder(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", f"{pipeline['database']['defaultTable']}_segments", 'sector', 'total', ['industry', 'transport'], 'housing')

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_segments',
 'rows': 183960,
 'columns': []}

In [23]:
read_sample(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", where="sector='housing'")

,timestamp,geography,sector,value
0,2024-01-01 00:00:00,01,housing,1279.739448
1,2024-01-01 01:00:00,01,housing,1276.873496
2,2024-01-01 02:00:00,01,housing,1270.062473
3,2024-01-01 03:00:00,01,housing,1262.919544
4,2024-01-01 04:00:00,01,housing,1271.694988
...,...,...,...,...
183955,2024-12-31 19:00:00,25,housing,1011.664716
183956,2024-12-31 20:00:00,25,housing,984.164233
183957,2024-12-31 21:00:00,25,housing,970.261122
183958,2024-12-31 22:00:00,25,housing,954.146784


In [21]:
segment_remove(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments", f"{pipeline['database']['defaultTable']}_segments", 'sector', 'total')

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'core_segments',
 'rows': -183960,
 'columns': []}

In [24]:
read_sample(output_path / pipeline['database']['file'], f"{pipeline['database']['defaultTable']}_segments")

,timestamp,geography,sector,value
0,2024-01-01 00:00:00,01,housing,1279.739448
1,2024-01-01 00:00:00,01,transport,243.483432
2,2024-01-01 00:00:00,01,industry,100.000000
3,2024-01-01 01:00:00,01,housing,1276.873496
4,2024-01-01 01:00:00,01,transport,242.977676
...,...,...,...,...
551875,2024-12-31 22:00:00,25,transport,186.025903
551876,2024-12-31 22:00:00,25,housing,954.146784
551877,2024-12-31 23:00:00,25,industry,100.000000
551878,2024-12-31 23:00:00,25,transport,183.748719


In [ ]:
# 4.2. Execute transformations - Apply growth over time

## Extract geographies
geographies = geography_demand.index.get_level_values('geography').unique()

## Define growth parameters
start_time = pd.Timestamp(config['start'])
end_time = pd.Timestamp(config['end'])

new_timestamps = pd.date_range(start=start_time, end=end_time, freq=config['baseResolution'])

## Extract hourly demand patterns for 2024
hourly_demand_2024 = (
    geography_demand.loc[
        geography_demand.index.get_level_values('timestamp').year == 2024
    ]
    .reset_index()
)

# Repeat the 2024 demand pattern to match the new timestamps
hourly_demand_pattern = hourly_demand_2024.groupby("geography")["demand"].apply(
    lambda x: np.tile(x.values, len(new_timestamps) // len(x) + 1)[:len(new_timestamps)]
)

# Convert hourly demand pattern back to a 2D array (municipalities x timestamps)
base_demand_repeated = np.vstack(hourly_demand_pattern.values)

## Get the scenarios
transformer = transformers[1]
transformer_scenarios = transformer['scenarios']

## Working array
extended_data = []

for transformer_scenario in transformer_scenarios:
    scenario_name = transformer_scenario['name']
    scenario_index = transformer_scenario['index']
    if transformer_scenario['type'] == 'exp-growth-to-target':
        scenario_target = transformer_scenario['target']
    scenario_randomness = transformer_scenario.get('randomness', 0)  # Default to 0 if not provided

    ## Generate the index and growth factors for the extended demand
    growth_factors = generate_growth_time_series(new_timestamps, config['baseResolution'], transformer_scenario)

    # Generate randomness
    random_factors = np.random.uniform(-scenario_randomness, scenario_randomness, size=base_demand_repeated.shape)

    # Apply growth factors
    extended_demand_values = base_demand_repeated * growth_factors * (1+random_factors)

    new_index = pd.MultiIndex.from_product(
        [geographies, new_timestamps, [scenario_index]], names=["geography", "timestamp", scenario_name]
    )

    # Create the extended demand DataFrame
    extended_geography_demand = pd.DataFrame(
        data=extended_demand_values.flatten(),
        index=new_index,
        columns=["demand"]
    )

    # Append to the list
    extended_data.append(extended_geography_demand)

extended_geography_scenario_demand = pd.concat(extended_data)

In [ ]:
# 0. Grab your configured baseline year
base_year = 2025  # or pull from your config.yaml

raw = extended_geography_scenario_demand['demand']
years = raw.index.get_level_values('timestamp').year
months = raw.index.get_level_values('timestamp').month

# 1. Compute a single‐year (July of base_year) industry baseline
industry = (
    raw
    .where((months == 7) & (years == base_year))       # only July of base_year
    .groupby([
        raw.index.get_level_values('geography'),
        raw.index.get_level_values('growth')
    ])
    .transform('min')                                  # lowest July of that year
    .mul(0.5)                                          # 50% of that
)

# 2. Compute remainder, buildings, transport
remainder = raw.sub(industry)
buildings = remainder.mul(0.95)
transport = remainder.mul(0.05)

# 3. Stack into long‐form Series
segmented = pd.concat(
    {'industry': industry, 'buildings': buildings, 'transport': transport},
    names=['segment']
)

# 4. Reset index & rename to match schema
extended_geography_scenario_sector_demand = (
    segmented
      .rename('value')
      .reset_index()
      .rename(columns={
          'timestamp': 'period.start',
          'geography': 'dimensions.geography',
          'segment':   'dimensions.segment.level1',
          'growth':    'scenario.growth'
      })
)


In [ ]:
''' OLD CODE
# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']

## TODO: Make this less naive

## This transform is not realistic. It does the following:
##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality
##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder 

## Extract July data for each year
july_data = extended_geography_scenario_demand.loc[
    extended_geography_scenario_demand.index.get_level_values('timestamp').month == 7
]

# Compute the lowest July value per municipality, year, and growth scenario
lowest_july_per_year = (
    july_data
    .groupby([
        july_data.index.get_level_values('geography'),
        july_data.index.get_level_values('timestamp').year,
        july_data.index.get_level_values('growth')  # Ensure growth scenario is part of grouping
    ])['demand']
    .min()
)

# Convert to DataFrame and calculate industry demand
lowest_july_per_year = lowest_july_per_year.to_frame(name='lowest_july')
lowest_july_per_year['industry_demand'] = lowest_july_per_year['lowest_july'] * 0.5

# Create extended_geography_sector_demand and merge industry demand back with the full extended demand
extended_geography_scenario_sector_demand = extended_geography_scenario_demand.copy()

# Extract year from timestamp in the index
extended_geography_scenario_sector_demand['year'] = extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year

# Join the calculated industry demand on ['geography', 'year', 'growth']
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.join(
    lowest_july_per_year['industry_demand'],
    on=['geography', 'year', 'growth']
)

# Industry demand is constant across each municipality and year
extended_geography_scenario_sector_demand['industry'] = extended_geography_scenario_sector_demand['industry_demand']

# Compute the remainder for buildings and transport
extended_geography_scenario_sector_demand['remainder'] = (
    extended_geography_scenario_sector_demand['demand'] - extended_geography_scenario_sector_demand['industry']
)

# Split remainder into buildings (95%) and transport (5%)
extended_geography_scenario_sector_demand['buildings'] = extended_geography_scenario_sector_demand['remainder'] * 0.95
extended_geography_scenario_sector_demand['transport'] = extended_geography_scenario_sector_demand['remainder'] * 0.05

# Drop unnecessary intermediate columns
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.drop(columns=['demand', 'industry_demand', 'remainder', 'year'])'
'''

" OLD CODE\n# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']\n\n## TODO: Make this less naive\n\n## This transform is not realistic. It does the following:\n##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality\n##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder \n\n## Extract July data for each year\njuly_data = extended_geography_scenario_demand.loc[\n    extended_geography_scenario_demand.index.get_level_values('timestamp').month == 7\n]\n\n# Compute the lowest July value per municipality, year, and growth scenario\nlowest_july_per_year = (\n    july_data\n    .groupby([\n        july_data.index.get_level_values('geography'),\n        july_data.index.get_level_values('timestamp').year,\n        july_data.index.get_level_values('growth')  # Ensure growth scenario is part of grouping\n    ])['demand']\n    .min()\n)\n\n# Convert to DataFrame and calculate in

**The code below all pertains to writing data.**

In [ ]:
## Clear the api folder
if clear_api_flag:
    print("Clearing API folder...")
    clear_api(data_path)

Clearing API folder...


In [ ]:
# Partition dataframe and write to parquet with pyarrow

use_scenario_id = config['generator']['useScenarioId']
partition_keys = config['generator']['partitionKeys']
row_group_size = config['generator']['rowGroupSize']

# 1) If scenario._id is used, compute and append it directly in the DataFrame
if use_scenario_id:
    print("Adding scenario._id to DataFrame...")

    scenario_cols = sorted(
        col for col in extended_geography_scenario_sector_demand.columns
        if col.startswith('scenario.')
    )

    extended_geography_scenario_sector_demand['scenario._id'] = (
        extended_geography_scenario_sector_demand[scenario_cols]
        .astype(str)
        .rename(columns=lambda col: col.split('.', 1)[1])  # remove "scenario." prefix
        .agg(lambda row: '+'.join(f"{k}:{v}" for k, v in row.items()), axis=1)
    )

# 2) Convert your pandas DF into an Arrow Table
print("Converting DataFrame to Arrow Table...")
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

actual_partitions = []

# 3) Compute derived keys and append
print("Computing derived partition keys...")
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]
        tf  = entry["transform"]
        arr = getattr(pc, tf)(table[src])
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    elif isinstance(entry, dict):
        actual_partitions.append(entry['path'])

# 4) Write Parquet dataset partitioned on actual columns
print("Writing Parquet dataset...")
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)


Adding scenario._id to DataFrame...
Converting DataFrame to Arrow Table...
Computing derived partition keys...
Writing Parquet dataset...


In [ ]:
extended_geography_scenario_sector_demand

,dimensions.segment.level1,dimensions.geography,period.start,scenario.growth,value,scenario._id
0,industry,01,2025-01-01 00:00:00+00:00,0,554.651596,growth:0
1,industry,01,2025-01-01 01:00:00+00:00,0,554.651596,growth:0
2,industry,01,2025-01-01 02:00:00+00:00,0,554.651596,growth:0
3,industry,01,2025-01-01 03:00:00+00:00,0,554.651596,growth:0
4,industry,01,2025-01-01 04:00:00+00:00,0,554.651596,growth:0
...,...,...,...,...,...,...
33135475,transport,25,2044-12-31 19:00:00+00:00,2,147.583130,growth:2
33135476,transport,25,2044-12-31 20:00:00+00:00,2,149.283653,growth:2
33135477,transport,25,2044-12-31 21:00:00+00:00,2,136.600535,growth:2
33135478,transport,25,2044-12-31 22:00:00+00:00,2,140.067808,growth:2


In [ ]:
# Partition dataframe and write to parquet with pyarrow (OLD)
'''
use_scenario_id = config['generator']['useScenarioId']
partition_keys = config['generator']['partitionKeys']
row_group_size = config['generator']['rowGroupSize']

# 1) Convert your pandas DF into an Arrow Table
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

# TODO: Rewrite a more elegant function for transforming the partitioning.yaml (maybe change the format to a schema)
# 2) Compute each derived key in Arrow and append as a real column
#    We’ll also build a list of the *actual* column names to partition on:
actual_partitions = []
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]            # e.g. "period.start"
        tf  = entry["transform"]       # e.g. "year"
        # Compute via pyarrow.compute.<tf>(...)
        arr = getattr(pc, tf)(table[src])
        # Append that as a column named exactly tf
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    else:
        # plain existing column
        actual_partitions.append(entry['path'])

# 3) Now write a Hive‐style dataset over those *real* columns
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,         # list[str] works with hive flavor
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)
'''